# Claude Structured-Output Evaluation

Evaluate a selected Claude model on Calculus I multiple-choice questions. Responses use Claude's native structured-output API and are checkpointed after each successful question so interrupted runs can resume.

## 1. Imports and configuration

In [1]:
import json
import os
import re
from pathlib import Path

import pandas as pd
from anthropic import Anthropic
from dotenv import load_dotenv
from pydantic import BaseModel
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
)

### Run configuration

Change only `MODEL_NAME` to test another Claude model. The artifact directory and every generated filename update automatically.

In [2]:
# Support launching Jupyter from either the repository root or notebooks/.
WORKING_DIRECTORY = Path.cwd()
PROJECT_ROOT = (
    WORKING_DIRECTORY.parent
    if WORKING_DIRECTORY.name == "notebooks"
    else WORKING_DIRECTORY
)

MODEL_NAME = "claude-opus-4-5-20251101"
RUN_NAME = f"{MODEL_NAME}-subsample"

DATA_PATH = PROJECT_ROOT / "data" / "test3_input_subset_seed42_numq1 (clean testing dataset).csv"
MODEL_OUTPUT_DIRECTORY = PROJECT_ROOT / "output" / MODEL_NAME
CHECKPOINT_PATH = MODEL_OUTPUT_DIRECTORY / f"{RUN_NAME}-checkpoint.json"
RESULTS_PATH = MODEL_OUTPUT_DIRECTORY / f"{RUN_NAME}-test-output.csv"
INCORRECT_RESULTS_PATH = (
    MODEL_OUTPUT_DIRECTORY / f"{RUN_NAME}-test-output-incorrect.csv"
)

MODEL_OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

## 2. Anthropic client and response schema

In [3]:
load_dotenv()
client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))


class Calculus(BaseModel):
    """Structured response for one calculus question."""

    detailed_solution_explanation: str
    correct_option_choice_letter: str

### Few-shot example

In [4]:
EXAMPLE_RESPONSE = '''
"Complete the square to write the equation in the standard form f(x)=a(x-h)^2+k. f(x)=2x^2+12x+25
                a) f(x)=2(x+3)^2+16
                b) f(x)=2(x-6)^2+7
                c) f(x)=2(x+3)^2+7
                d) f(x)=2(x-3)^2+16
                e) f(x)=2(x+3)^2-7
                f) None of the above."
    "detailed_solution_explanation": "To complete the square for f(x) = 2x² + 12x + 25, first factor out 2 from the terms involving x:\n\nf(x) = 2(x² + 6x) + 25.\n\nNext, complete the square inside the parentheses:\n\nx² + 6x = x² + 6x + 9 − 9 = (x + 3)² − 9.\n\nRewrite f(x):\n\nf(x) = 2[(x + 3)² − 9] + 25\n      = 2(x + 3)² − 18 + 25\n      = 2(x + 3)² + 7.\n",
    "correct_option_choice_letter": "c"
'''

SYSTEM_PROMPT = (
    "Act as a Mathematician specializing in Calculus. Solve the given "
    "college-level Calculus 1 multiple-choice problem, using the given example "
    "as a reference. Provide a step-by-step explanation of the solution process, "
    "clearly outlining the reasoning behind each step. Choose the correct letter "
    "option for the given problem. Ensure that the solutions are accurate, "
    "comprehensive, and easy to understand. Here's the example: "
    + EXAMPLE_RESPONSE
)

## 3. Load the dataset

In [5]:
def load_dataset(data_path: Path) -> pd.DataFrame:
    """Load the evaluation dataset."""
    return pd.read_csv(data_path)


df = load_dataset(DATA_PATH)
print(f"Loaded {len(df)} questions from {DATA_PATH}")
df.head()

Loaded 331 questions from /Users/mansibmursalin/Library/CloudStorage/OneDrive-UniversityOfHouston/CASA LLM Project/Publishable Project Repo/data/test3_input_subset_seed42_numq1 (clean testing dataset).csv


,ID,Question,Correct Answer
0,q2_sp22_problem10__04,4) Complete the square to write the equation i...,b
1,q27_sp23_problem04__04,4) A rectangular playground is to be fenced of...,b
2,q23_sp23_problem09__02,2) Differentiate \(\displaystyle f(x)=\tan \le...,b
3,q31_sp23_problem03__09,9) Given that \(x &gt; 0\) and \( \text{capit...,d
4,q10_sp22_problem07__21,21) Find the slope of the line that is tangent...,c


## 4. Response pipeline

Each successfully parsed response is written to the model-specific checkpoint immediately.

In [6]:
def load_checkpoint(checkpoint_path: Path) -> dict:
    """Load saved responses, or return an empty checkpoint."""
    if not checkpoint_path.exists():
        return {}
    with checkpoint_path.open("r", encoding="utf-8") as checkpoint_file:
        return json.load(checkpoint_file)


def save_checkpoint(checkpoint_path: Path, data: dict) -> None:
    """Persist responses as formatted JSON."""
    with checkpoint_path.open("w", encoding="utf-8") as checkpoint_file:
        json.dump(data, checkpoint_file, indent=4)

In [14]:
def query_claude(question: str, model: str = MODEL_NAME) -> dict:
    """Request one structured calculus solution from Claude."""
    try:
        response = client.messages.parse(
            model=model,
            max_tokens=4096,
            system=SYSTEM_PROMPT,
            messages=[
                {
                    "role": "user",
                    "content": "Now solve the following question: " + question,
                }
            ],
            output_format=Calculus,
        )
    except Exception as error:
        print(f"Error querying Claude: {error}")
        return {}

    parsed_output = response.parsed_output
    return parsed_output.model_dump() if parsed_output else {}

In [15]:
def collect_responses(
    dataframe: pd.DataFrame,
    checkpoint_path: Path,
    model: str = MODEL_NAME,
) -> dict:
    """Query unanswered rows and update the checkpoint incrementally."""
    checkpoint = load_checkpoint(checkpoint_path)

    for _, row in dataframe.iterrows():
        question_id = str(row["ID"])
        if question_id in checkpoint:
            print(f"Skipping ID {question_id}, already processed.")
            continue

        print(f"Processing ID {question_id}...")
        response = query_claude(row["Question"], model=model)
        if not response:
            print(f"Failed to get response for {question_id}")
            continue

        checkpoint[question_id] = response
        save_checkpoint(checkpoint_path, checkpoint)

    save_checkpoint(checkpoint_path, checkpoint)
    print(f"Checkpoint saved to {checkpoint_path}")
    return checkpoint

In [18]:
def add_model_responses(dataframe: pd.DataFrame, checkpoint: dict) -> pd.DataFrame:
    """Add answer choices and explanations from the checkpoint."""
    result = dataframe.copy()
    question_ids = result["ID"].astype(str)
    result["LLM Answer Choice"] = question_ids.map(
        lambda question_id: checkpoint.get(question_id, {}).get(
            "correct_option_choice_letter", ""
        )
    )
    result["LLM Answer Explanation"] = question_ids.map(
        lambda question_id: checkpoint.get(question_id, {}).get(
            "detailed_solution_explanation", ""
        )
    )
    return result

### Run the response pipeline

> **Note:** This cell makes billable Anthropic API calls only for questions absent from the checkpoint.

In [19]:
df = df.head(10)  # Limit to first 10 rows for testing

checkpoint = collect_responses(df, CHECKPOINT_PATH)
df = add_model_responses(df, checkpoint)
df.head(10)

Skipping ID q2_sp22_problem10__04, already processed.
Skipping ID q27_sp23_problem04__04, already processed.
Skipping ID q23_sp23_problem09__02, already processed.
Skipping ID q31_sp23_problem03__09, already processed.
Skipping ID q10_sp22_problem07__21, already processed.
Skipping ID q19_fa22_problem03__21, already processed.
Skipping ID q28_sp23_problem03__04, already processed.
Skipping ID q25_sp23_problem08__02, already processed.
Skipping ID q2_sp22_problem06__04, already processed.
Skipping ID q5_sp22_problem07__01, already processed.
Checkpoint saved to /Users/mansibmursalin/Library/CloudStorage/OneDrive-UniversityOfHouston/CASA LLM Project/Publishable Project Repo/output/claude-opus-4-5-20251101/claude-opus-4-5-20251101-subsample-checkpoint.json


,ID,Question,Correct Answer,LLM Answer Choice (RAW),LLM Answer Explanation,LLM Answer Choice,Correct
0,q2_sp22_problem10__04,4) Complete the square to write the equation i...,b,b,To complete the square for f(x) = 4x² - 48x + ...,b,0
1,q27_sp23_problem04__04,4) A rectangular playground is to be fenced of...,b,b,Let the length of the playground be L and the ...,b,0
2,q23_sp23_problem09__02,2) Differentiate \(\displaystyle f(x)=\tan \le...,b,b,"To differentiate f(x) = tan(5^(6x²)), we need ...",b,0
3,q31_sp23_problem03__09,9) Given that \(x &gt; 0\) and \( \text{capit...,d,d,"To find F'(x), we apply the Fundamental Theore...",d,0
4,q10_sp22_problem07__21,21) Find the slope of the line that is tangent...,c,c,To find the slope of the tangent line to the g...,c,0
5,q19_fa22_problem03__21,21) Find the intervals on which \(f(x)=\displa...,c,c,"To find where f(x) = 6x/(x² + 4) decreases, we...",c,0
6,q28_sp23_problem03__04,4) Use differentials to estimate the value of ...,e,e,"To estimate (80.5)^(1/4) using differentials, ...",e,0
7,q25_sp23_problem08__02,2) Differentiate the following: \(f(x)=\displ...,b,b,"To differentiate f(x) = ln(arctan(5x + 6)), we...",b,0
8,q2_sp22_problem06__04,4) Find the <i>x</i>- and <i>y</i>-intercepts ...,b,b,To find the x- and y-intercepts of f(x) = x² -...,b,0
9,q5_sp22_problem07__01,1) Evaluate \(\displaystyle \lim _{x\rightarro...,c,c,To evaluate the limit of a piecewise function ...,c,0


## 5. Normalize and evaluate answers

In [20]:
def extract_letter_choice(text: str) -> str:
    """Return the first alphabetic character from a raw answer choice."""
    match = re.search(r"[a-zA-Z]", text)
    return match.group(0) if match else ""


def evaluate_answers(dataframe: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Normalize predictions and calculate weighted classification metrics."""
    result = dataframe.copy()
    result["LLM Answer Choice"] = result["LLM Answer Choice (RAW)"].map(
        extract_letter_choice
    )
    result["Correct"] = (
        result["Correct Answer"] == result["LLM Answer Choice"]
    ).astype(int)

    y_true = result["Correct Answer"]
    y_pred = result["LLM Answer Choice"]
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "classification_report": classification_report(
            y_true, y_pred, zero_division=0
        ),
    }
    return result, metrics

In [21]:
df, metrics = evaluate_answers(df)

print(metrics["classification_report"])
print(f"Accuracy:  {metrics['accuracy'] * 100:.2f}%")
print(f"Precision: {metrics['precision'] * 100:.2f}%")
print(f"Recall:    {metrics['recall'] * 100:.2f}%")
print(f"F1 Score:  {metrics['f1'] * 100:.2f}%")
df.head()

              precision    recall  f1-score   support

           b       1.00      1.00      1.00         5
           c       1.00      1.00      1.00         3
           d       1.00      1.00      1.00         1
           e       1.00      1.00      1.00         1

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10

Accuracy:  100.00%
Precision: 100.00%
Recall:    100.00%
F1 Score:  100.00%


,ID,Question,Correct Answer,LLM Answer Choice (RAW),LLM Answer Explanation,LLM Answer Choice,Correct
0,q2_sp22_problem10__04,4) Complete the square to write the equation i...,b,b,To complete the square for f(x) = 4x² - 48x + ...,b,1
1,q27_sp23_problem04__04,4) A rectangular playground is to be fenced of...,b,b,Let the length of the playground be L and the ...,b,1
2,q23_sp23_problem09__02,2) Differentiate \(\displaystyle f(x)=\tan \le...,b,b,"To differentiate f(x) = tan(5^(6x²)), we need ...",b,1
3,q31_sp23_problem03__09,9) Given that \(x &gt; 0\) and \( \text{capit...,d,d,"To find F'(x), we apply the Fundamental Theore...",d,1
4,q10_sp22_problem07__21,21) Find the slope of the line that is tangent...,c,c,To find the slope of the tangent line to the g...,c,1


## 6. Export results

In [22]:
incorrect_df = df.loc[df["Correct"] == 0].copy()

incorrect_df.to_csv(INCORRECT_RESULTS_PATH, index=False)
df.to_csv(RESULTS_PATH, index=False)

print(f"Saved {len(df)} results to {RESULTS_PATH}")
print(f"Saved {len(incorrect_df)} incorrect results to {INCORRECT_RESULTS_PATH}")

Saved 10 results to /Users/mansibmursalin/Library/CloudStorage/OneDrive-UniversityOfHouston/CASA LLM Project/Publishable Project Repo/output/claude-opus-4-5-20251101/claude-opus-4-5-20251101-subsample-test-output.csv
Saved 0 incorrect results to /Users/mansibmursalin/Library/CloudStorage/OneDrive-UniversityOfHouston/CASA LLM Project/Publishable Project Repo/output/claude-opus-4-5-20251101/claude-opus-4-5-20251101-subsample-test-output-incorrect.csv
